# message的使用

## 1、创建message

In [1]:
import os

from debugpy._vendored.pydevd.pydevd_attach_to_process.winappdbg import system
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_classic import model_laboratory
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from openai import api_key, base_url
from openrouter.errors.openrouterdefaulterror import MAX_MESSAGE_LEN

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)


In [2]:
# json格式的消息

messages = [
    {"role": "system", "content": "你是一个友好的AI助手"},
    {"role": "user", "content": "1 + 2 = ?"},
    {"role": "assistant", "content": "3"},
    {"role": "user", "content": "我刚才问了什么问题"},
]
response = model.invoke(messages)
print(response)

content='你刚才问了“1 + 2 = ?”，我回答了3。' additional_kwargs={'refusal': None, 'reasoning_content': '我们注意到用户问的是“我刚才问了什么问题”。这是一个关于对话历史的问题。在之前的对话中，用户问的是“1 + 2 = ?”，我回答了“3”。现在用户问“我刚才问了什么问题”，我需要根据对话历史回答。所以应该回答“你刚才问了‘1 + 2 = ?’”。'} response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 23, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 67, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 23}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '4c211ee6-e295-4483-8346-2f7cad6f3e67', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fb22e-a3db-7eb1-a9ab-e331a375bfde-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 23, 'output_tokens': 83, 'to

举例2:消息对象

In [4]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
messages = [
    SystemMessage(content="你是一个友好的AI助手"),
    HumanMessage(content="1 + 2 = ?"),
    AIMessage(content="3"),
    HumanMessage(content="我刚才问了什么问题"),
]
response = model.invoke(messages)
print(response.content)

你刚才问的问题是：“1 + 2 = ?”


## HumanMessage的使用
举例

In [5]:

messages = [
 SystemMessage("你是一个信息抽取器。你会收到多条来自不同发言者的 user 消息。每条消息可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出格式：{\"speakers\":[{\"name\":\"...\",\"claim\":\"...\"}]}"),
 HumanMessage(
    content="我认为 1+1=2",
    name="Bob"
    ),
 HumanMessage(
    content="我认为 1+1>2",
    name="Tom"
    ),
 HumanMessage(
    content="请列出谁说了什么，不要判断对错。",
    name="audience"
 )
]
response = model.invoke(messages)
print(response.content)

{"speakers":[{"name":"unknown","claim":"1+1=2"},{"name":"unknown","claim":"1+1>2"}]}


## ToolMessage的使用

举例1

In [17]:
import os

from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.messages.tool import ToolMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = ChatDeepSeek(
    model="deepseek-v4-pro",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
)
# ===== 让 Deepseek API 收到 reasoning_content =====
_original_get_request_payload = ChatDeepSeek._get_request_payload

def _patched_get_request_payload(self, input_, *, stop=None, **kwargs):
    payload = _original_get_request_payload(self, input_, stop=stop, **kwargs)
    messages = self._convert_input(input_).to_messages()
    for i, msg in enumerate(payload.get("messages", [])):
        if i < len(messages) and isinstance(messages[i], AIMessage):
            rc = messages[i].additional_kwargs.get("reasoning_content")
            if rc:
                msg["reasoning_content"] = rc
    return payload

ChatDeepSeek._get_request_payload = _patched_get_request_payload
# ====================================================================

def get_weather(city: str) -> str:
    return "不错哦~"

# 模拟模型绑定工具
# model_with_tools = model.bind_tools([get_weather])
ai_message = AIMessage(
    content="",
    additional_kwargs= {"reasoning_content": "用户问的是天气情况...我先运用工具查询..."},  # 传回思考过程
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "北京"},
        "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
    }]
)
tool_message = ToolMessage(
    content = "今天北京天气晴朗，万里无云~",
    tool_call_id = "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
)
messages = [
    HumanMessage(content="北京天气如何"),
    ai_message,
    tool_message
]

response = model.invoke(messages)
print(response)


content='今天北京天气晴朗，万里无云，适合出门走走~' additional_kwargs={'refusal': None, 'reasoning_content': '根据查询结果，今天北京天气晴朗，可以直接告诉用户。不需要再补充其他信息。'} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 84, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 19, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 84}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-pro', 'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402', 'id': 'e67d553c-d8df-4ccd-8bfc-f85428181a2a', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fb26f-3ab3-7f12-9f7d-b1ee663ad4e0-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 84, 'output_tokens': 33, 'total_tokens': 117, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reaso

### 对话历史优化

In [2]:
import os

from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = ChatDeepSeek(
    model="deepseek-v4-pro",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
)


def keep_recent_messages(messages, max_pairs=3):
    """
    保留最近的 N 轮对话
    max_pairs: 保留的对话轮数（每轮 = user + assistant）
    """
    # 分离 system 和对话
    system_messags = [m for m in messages if m["role"] == "system"]
    conversation_messages = [m for m in messages if m["role"] != "system"]
    # 只保留最近的 N 轮对话
    return system_messags + conversation_messages[-(max_pairs * 2):]

# 初始化
long_conversation = [
    {"role": "system", "content": "你是python导师"}
]

# 第 1 轮
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})

# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})

# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})
print(f"原始消息数: {len(long_conversation)}")

# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")
print(f"保留的内容: system + 最近2轮对话")

# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么？"})
# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

原始消息数: 7
优化后消息数: 5
保留的内容: system + 最近2轮对话

AI 回复: 你第一个问题是：**“列表和元组有什么区别？用一句解释”** 对吗？


## 多轮对话聊天机器人

In [5]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model= "deepseek-v4-flash",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL
)

MAX_MESSAGE_HISTORY = 10 # 对话历史长度
EXIT_WORD = "quit"

messages = [{"role": "system", "content": "你是小谷姐姐，一个 helpful 的助手"}]

i = 1
while True:
    print("\n","="*10,f"第 {i} 轮对话开始", "="*10,"\n")
    user_input = input("请输入：")

    if user_input == EXIT_WORD:
        print("对话结束")
        break

    messages.append({"role": "user", "content": user_input})

    print("小谷姐姐：",end="",flush=True)

    ai_message = ""

    # 优化历史记忆
    messages = keep_recent_messages(messages,max_pairs=MAX_MESSAGE_HISTORY)

    for chunk in model.stream( messages):
        if chunk.content:
           ai_message += chunk.content
           print(chunk.content, end="", flush=True)

    messages.append({"role": "assistant", "content": ai_message})

    print("\n","="*10,f"第 {i} 轮对话结束", "="*10,"\n")

    i += 1


 ========== 第 1 轮对话开始 ========== 

小谷姐姐：你好呀！我是小谷姐姐，有什么可以帮你的吗？😊 无论是问题咨询、聊天解闷，还是需要帮忙，都可以告诉我哦～
 ========== 第 1 轮对话结束 ========== 


 ========== 第 2 轮对话开始 ========== 

小谷姐姐：中国目前有**23个省**，另外还有**5个自治区**、**4个直辖市**和**2个特别行政区**，所以全国共有**34个省级行政区**。

简单来说：
- **省（23个）**：河北、山西、辽宁、吉林、黑龙江、江苏、浙江、安徽、福建、江西、山东、河南、湖北、湖南、广东、海南、四川、贵州、云南、陕西、甘肃、青海、台湾。
- 其他：内蒙古、广西、西藏、宁夏、新疆（自治区）；北京、天津、上海、重庆（直辖市）；香港、澳门（特别行政区）。

如果需要更详细的，随时问我哦~
 ========== 第 2 轮对话结束 ========== 


 ========== 第 3 轮对话开始 ========== 

对话结束
